# MCLDNN Differential Attention Training and Comparison

This notebook trains the new `mcldnn_diffattention` model and compares it against the normal `mcldnn_attention` model.

Main rule followed here:

- Differential-attention results are exported/downloaded.
- Normal-attention results are used only for comparison plots/tables and are not included in the final zip.

The comparison uses:

- test accuracy
- accuracy vs SNR
- mean accuracy over all SNRs
- peak SNR accuracy
- parameter count

In [ ]:
# CELL 1: Setup repo and paths
import os
import sys
import shutil
import subprocess
from pathlib import Path
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from IPython.display import display, Image, FileLink

os.environ["KERAS_BACKEND"] = "tensorflow"

REPO_URL = "https://github.com/akshlabh/amr-5-class.git"
WORK_DIR = Path("/kaggle/working/amr-5-class")
DATASET = Path("/kaggle/input/datasets/gustavopolicarpo/rml201610a-dict/RML2016.10a_dict.dat")

# If internet is available, clone/pull. If not, use an already attached repo copy if present.
def find_attached_repo():
    for root in Path("/kaggle/input").glob("**"):
        if (root / "src" / "train.py").exists() and (root / "configs").exists():
            return root
    return None

if (WORK_DIR / ".git").exists():
    subprocess.run(["git", "-C", str(WORK_DIR), "pull"], check=False)
elif WORK_DIR.exists() and (WORK_DIR / "src" / "train.py").exists():
    print(f"Using existing work dir: {WORK_DIR}")
else:
    attached = find_attached_repo()
    if attached is not None:
        print(f"Copying attached repo from: {attached}")
        if WORK_DIR.exists():
            shutil.rmtree(WORK_DIR)
        shutil.copytree(attached, WORK_DIR)
    else:
        print("Cloning repo from GitHub...")
        subprocess.run(["git", "clone", REPO_URL, str(WORK_DIR)], check=True)

os.chdir(WORK_DIR)
if str(WORK_DIR) not in sys.path:
    sys.path.insert(0, str(WORK_DIR))

print(f"Working dir: {Path.cwd()}")
print(f"Dataset    : {DATASET}")
print(f"Dataset OK : {DATASET.exists()}")

required = [
    "src/models/mcldnn_attention.py",
    "src/models/mcldnn_diffattention.py",
    "configs/exp_5class_attention.yaml",
    "configs/exp_5class_diffattention.yaml",
    "src/train.py",
]
for f in required:
    print(f"{'?' if Path(f).exists() else '? MISSING'} {f}")

In [ ]:
# CELL 2: Quick model shape and parameter check
import gc
import keras
import keras.backend as K

keras.mixed_precision.set_global_policy("float32")
K.clear_session(); gc.collect()

from src.models.mcldnn_attention import build_mcldnn_attention
from src.models.mcldnn_diffattention import (
    build_mcldnn_diffattention,
    build_mcldnn_diffattention_extractor,
)

normal_model = build_mcldnn_attention(classes=5)
diff_model = build_mcldnn_diffattention(classes=5)
diff_extractor = build_mcldnn_diffattention_extractor(classes=5)

print(f"Normal attention params : {normal_model.count_params():,}")
print(f"Diff attention params   : {diff_model.count_params():,}")
print(f"Under 300k              : {'YES' if diff_model.count_params() < 300_000 else 'NO'}")

x1 = np.zeros((4, 2, 128, 1), dtype="float32")
x2 = np.zeros((4, 128, 1), dtype="float32")
x3 = np.zeros((4, 128, 1), dtype="float32")

pred = diff_model.predict([x1, x2, x3], verbose=0)
pred2, attn = diff_extractor.predict([x1, x2, x3], verbose=0)

print(f"Diff training output shape  : {pred.shape}")
print(f"Diff extractor softmax shape: {pred2.shape}")
print(f"Diff attention map shape    : {attn.shape}")
print(f"Diff attention min/max      : {attn.min():.5f}, {attn.max():.5f}")

assert pred.shape == (4, 5)
assert pred2.shape == (4, 5)
assert attn.shape == (4, 2, 124, 124)
assert diff_model.count_params() < 300_000

print("Shape checks passed")

del normal_model, diff_model, diff_extractor
K.clear_session(); gc.collect()

In [ ]:
# CELL 3: Train Differential Attention model
# Outputs go only under experiments/5class_diffattention.
assert DATASET.exists(), f"Dataset not found: {DATASET}"

subprocess.run([
    "python", "src/train.py",
    "--config", "configs/exp_5class_diffattention.yaml",
    "--datasetpath", str(DATASET),
], check=True)

print("\nDifferential attention training finished.")
print("Checkpoint:", Path("experiments/5class_diffattention/checkpoints/best_model.weights.h5").exists())
print("Test score:", Path("experiments/5class_diffattention/results/test_score.csv").exists())

In [ ]:
# CELL 4: Ensure normal attention results are available for comparison
# This cell trains normal attention only if its result CSV is missing.
# Normal-attention outputs are NOT included in the final download zip.

NORMAL_RESULT = Path("experiments/5class_attention/results/test_score.csv")
NORMAL_SNR = Path("experiments/5class_attention/results/acc_per_snr.csv")

if NORMAL_RESULT.exists() and NORMAL_SNR.exists():
    print("Normal attention results already exist. Using them for comparison only.")
else:
    print("Normal attention results not found. Training normal attention for comparison only...")
    subprocess.run([
        "python", "src/train.py",
        "--config", "configs/exp_5class_attention.yaml",
        "--datasetpath", str(DATASET),
    ], check=True)

print("Normal attention test score exists:", NORMAL_RESULT.exists())
print("Normal attention SNR CSV exists  :", NORMAL_SNR.exists())

In [ ]:
# CELL 5: Compare Normal Attention vs Differential Attention
NORMAL_DIR = Path("experiments/5class_attention")
DIFF_DIR = Path("experiments/5class_diffattention")
COMPARE_DIR = DIFF_DIR / "comparison_with_normal_attention"
COMPARE_DIR.mkdir(parents=True, exist_ok=True)

normal_score = pd.read_csv(NORMAL_DIR / "results" / "test_score.csv")
diff_score = pd.read_csv(DIFF_DIR / "results" / "test_score.csv")

normal_snr = pd.read_csv(NORMAL_DIR / "results" / "acc_per_snr.csv")
diff_snr = pd.read_csv(DIFF_DIR / "results" / "acc_per_snr.csv")

normal_snr = normal_snr.rename(columns={"accuracy": "normal_attention_acc"})
diff_snr = diff_snr.rename(columns={"accuracy": "diff_attention_acc"})
comparison = pd.merge(normal_snr, diff_snr, on="snr", how="inner")
comparison["delta_diff_minus_normal"] = comparison["diff_attention_acc"] - comparison["normal_attention_acc"]
comparison["normal_attention_acc_percent"] = 100.0 * comparison["normal_attention_acc"]
comparison["diff_attention_acc_percent"] = 100.0 * comparison["diff_attention_acc"]
comparison["delta_percent_points"] = 100.0 * comparison["delta_diff_minus_normal"]

summary = pd.DataFrame([
    {
        "model": "normal_attention",
        "test_loss": float(normal_score.loc[0, "loss"]),
        "test_accuracy": float(normal_score.loc[0, "accuracy"]),
        "mean_snr_accuracy": float(comparison["normal_attention_acc"].mean()),
        "peak_snr_accuracy": float(comparison["normal_attention_acc"].max()),
        "peak_snr_db": int(comparison.loc[comparison["normal_attention_acc"].idxmax(), "snr"]),
    },
    {
        "model": "diff_attention",
        "test_loss": float(diff_score.loc[0, "loss"]),
        "test_accuracy": float(diff_score.loc[0, "accuracy"]),
        "mean_snr_accuracy": float(comparison["diff_attention_acc"].mean()),
        "peak_snr_accuracy": float(comparison["diff_attention_acc"].max()),
        "peak_snr_db": int(comparison.loc[comparison["diff_attention_acc"].idxmax(), "snr"]),
    },
])

comparison_csv = COMPARE_DIR / "normal_vs_diffattention_acc_per_snr.csv"
summary_csv = COMPARE_DIR / "normal_vs_diffattention_summary.csv"
comparison.to_csv(comparison_csv, index=False)
summary.to_csv(summary_csv, index=False)

print("Summary:")
display(summary)
print("\nPer-SNR comparison:")
display(comparison)
print(f"Saved: {comparison_csv}")
print(f"Saved: {summary_csv}")

In [ ]:
# CELL 6: Plot comparison curves
fig, ax = plt.subplots(figsize=(11, 6))
ax.plot(comparison["snr"], comparison["normal_attention_acc_percent"],
        marker="o", linewidth=2.4, label="Normal attention")
ax.plot(comparison["snr"], comparison["diff_attention_acc_percent"],
        marker="s", linewidth=2.4, label="Differential attention")
ax.set_title("Normal Attention vs Differential Attention: Accuracy vs SNR")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Accuracy (%)")
ax.grid(True, alpha=0.3)
ax.legend()
ax.set_xticks(comparison["snr"])
plt.tight_layout()
curve_path = COMPARE_DIR / "normal_vs_diffattention_acc_vs_snr.png"
fig.savefig(curve_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {curve_path}")

fig, ax = plt.subplots(figsize=(11, 4.8))
colors = ["#2ca02c" if x >= 0 else "#d62728" for x in comparison["delta_percent_points"]]
ax.bar(comparison["snr"].astype(str), comparison["delta_percent_points"], color=colors)
ax.axhline(0, color="black", linewidth=1)
ax.set_title("Differential Attention minus Normal Attention by SNR")
ax.set_xlabel("SNR (dB)")
ax.set_ylabel("Delta accuracy (percentage points)")
ax.grid(True, axis="y", alpha=0.3)
plt.tight_layout()
delta_path = COMPARE_DIR / "diff_minus_normal_delta_by_snr.png"
fig.savefig(delta_path, dpi=180, bbox_inches="tight")
plt.show()
print(f"Saved: {delta_path}")

In [ ]:
# CELL 7: Copy comparison files into diff-attention results folder
# They are saved under the diff-attention experiment so the final zip is self-contained.
# Normal attention checkpoints/results are not copied.

DIFF_COMPARE_EXPORT = DIFF_DIR / "results" / "comparison_with_normal_attention"
DIFF_COMPARE_EXPORT.mkdir(parents=True, exist_ok=True)

for src in [comparison_csv, summary_csv, curve_path, delta_path]:
    dst = DIFF_COMPARE_EXPORT / Path(src).name
    shutil.copy2(src, dst)
    print(f"Copied: {dst}")

In [ ]:
# CELL 8: Download ONLY differential-attention results
# This zip includes experiments/5class_diffattention only.
# It does not include experiments/5class_attention normal-attention outputs.

stamp = datetime.now().strftime("%Y%m%d_%H%M")
zip_base = Path("/kaggle/working") / f"diffattention_results_{stamp}"
zip_path = shutil.make_archive(str(zip_base), "zip", root_dir=str(DIFF_DIR))

print(f"Created zip: {zip_path}")
print("\nIncluded files from differential-attention experiment:")
for path in sorted(DIFF_DIR.rglob("*")):
    if path.is_file():
        print(" -", path.relative_to(DIFF_DIR))

display(FileLink(zip_path))